In [4]:
from dotenv import load_dotenv
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
import asyncio
import os
from typing import Dict
from IPython.display import Markdown, display
from sendgrid.helpers.mail import Mail, Email, To, Content
import sendgrid

In [5]:
load_dotenv(override=True)

sendgrid_api_key = os.getenv("SENDGRID_API_KEY")

if sendgrid_api_key:
    print(f"Sendgrid API Key exists and begins {sendgrid_api_key[:6]}")
else:
    print(f"Sendgrid API Key is not set")

Sendgrid API Key exists and begins SG.vxq


In [6]:
INSTRUCTIONS = "You write concise research summary. 2~3 paragraphs, less than 300 words.\
               Capture main points, no fluff, no commentary"

search_agent = Agent(
    name="Search Agent",
    instructions=INSTRUCTIONS,
    tools= [WebSearchTool(search_context_size="low")],
    model= "gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required")
)

In [7]:
class WebSearchItem(BaseModel):
    reason: str=Field(description="Your reason for why this search is important to this query")
    query: str=Field(description="The search term to use for the web search")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem]=Field(description="A list of web searches to perform to best answer the query")

In [11]:
HOW_MANY_SEARCHES = 5

INSTRUCTIONS = f"You are a helpful research assistant agent. \
               When given a query, come up with a set of web searches to perform \
               to best answer the query. Output {HOW_MANY_SEARCHES} search terms to query for."

planner_agent = Agent(
    name="Planner Agent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan
)

In [12]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(sendgrid_api_key)
    from_email = Email("mhda08@naver.com") 
    to_email = To("mhda08@naver.com") 
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [16]:
INSTRUCTIONS = "You format and send HTML email. When receiving detailed report, use send_email tool\
                to provide appropriate subject line"

email_agent = Agent(
    name= "Email Agent",
    instructions= INSTRUCTIONS,
    tools= [send_email],
    model= "gpt-4o-mini"
)

In [14]:
INSTRUCTIONS = "You are a senior researcher, that writes cohesive detailed report.\
               First outline then full report using markdown format, 1000+ words"

class ReportData(BaseModel):
    short_summary: str=Field(description="A short 2~3 sentence summary of your findings")
    markdown_report: str = Field(description="Write a final report in Markdown")
    follow_up_questions: list[str] = Field(description="Add follow up questions for the user using the final report")

writer_agent = Agent(
    name="Writer Agent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData
)

In [15]:
class Criticism(BaseModel):
    is_approved: bool = Field(description="True if the report meets all quality standards, False otherwise.")
    critique_points: list[str] = Field(description="Specific, numbered list of improvements needed.")
    rating: int = Field(description="A quality score from 1 to 10.")

CRITIC_INSTRUCTIONS = """
You are a demanding Senior Editor. Your goal is to ensure research reports are 
top-tier, professional, and factually grounded. 

Review the report for:
1. DEPTH: Is it truly 1000+ words and detailed?
2. ACCURACY: Does it stay true to the provided search results?
3. STRUCTURE: Does it have clear headings, intro, and conclusion?
4. FOLLOW-UPS: Are the suggested further research topics insightful?

If the report is excellent, set is_approved to True. If not, be specific about what is missing.
"""

critic_agent = Agent(
    name="CriticAgent",
    instructions=CRITIC_INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=Criticism,
)

In [19]:
async def plan_searches(query:str):
    """Use the planner agent to plan which searches to run for the query"""
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query:{query}")
    print(f"Perform {len(result.final_output.searches)} searches")
    return result.final_output
    

async def search(item: WebSearchItem):
    input = f"Search term: {item.query}\nReason: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output  # one summary string


async def perform_searches(search_plan):
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    return results  # list of 5 summaries

In [20]:
async def write_report_with_reflection(query: str, search_results: list[str]):
    print("Writing initial draft...")
    input_text = f"Original query: {query}\nSummarized search results: {search_results}"
    
    # Generate the first draft
    current_report = await Runner.run(writer_agent, input_text)
    
    max_iterations = 3
    for i in range(max_iterations):
        print(f"Reflection Loop {i+1}: Critiquing report...")
        
        # Get feedback from the critic
        review = await Runner.run(critic_agent, current_report.final_output.markdown_report)
        
        if review.final_output.is_approved:
            print(f"Report approved with a score of {review.final_output.rating}/10!")
            break
        else:
            print(f"Feedback received: {review.final_output.critique_points}")
            print("Refining report based on feedback...")
            
            # Feed the critique back to the writer
            refinement_input = (
                f"Original query: {query}\n"
                f"Search results: {search_results}\n"
                f"Previous draft: {current_report.final_output.markdown_report}\n"
                f"Your previous draft was rejected.\n"
                f"Critique: {review.final_output.critique_points}\n"
                f"Please rewrite the report addressing all critique points."
            )
            current_report = await Runner.run(writer_agent, refinement_input)

    return current_report.final_output

In [25]:
async def send_report(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

### Put then all together!!!

In [26]:
query ="Latest AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report_with_reflection(query, search_results)
    await send_report(report)  
    print("Hooray!")




Starting research...
Planning searches...
Perform 5 searches
Writing initial draft...
Reflection Loop 1: Critiquing report...
Feedback received: ['The report is not 1000+ words; it is approximately 830 words long, falling short on depth.', 'While it references various sources, it lacks in-depth analysis and discussion on each framework and trend.', 'The structure should have more distinct sections within the trends for clarity. Consider including subheadings to separate different aspects of each trend.', "The follow-up section on 'Industry Conferences' lacks depth; further research topics should suggest emerging technologies or challenges within AI frameworks. More insightful suggestions would be beneficial."]
Refining report based on feedback...
Reflection Loop 2: Critiquing report...
Feedback received: ['The report is missing key subsections under the main headings, which would improve clarity and flow.', 'Some sections need expanded explanations or examples to reach the depth expect